# Pokemon Data Analysis

Using SQL & Python to analyse Pokemon data.


In [2]:
import pandas as pd
import duckdb

PATH_TO_CSV = "data/pokemon.csv"
pokemons = pd.read_csv(PATH_TO_CSV)

duckdb.sql("SELECT * FROM pokemons LIMIT 5").df()

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,5,80,65,80,fire,NaN,19.0,1,0


## Analysis


### Strongest Pokemons

To get the strongest Pokemons of each generation, ranked by the total of their base stats (HP, attack, defence, special attack, special defence, and speed), I've opted to go for a windows function


In [ ]:
QUERY = """
WITH cte AS (
    SELECT 
        generation,
        COALESCE(hp, 0) 
            + COALESCE(attack, 0) 
            + COALESCE(defense, 0) 
            + COALESCE(sp_attack, 0) 
            + COALESCE(sp_defense, 0) 
            + COALESCE(speed, 0) 
            AS base_stats_total,
        RANK() OVER (
            PARTITION BY generation
            ORDER BY base_stats_total DESC
        ) AS rank,
        name,
        CASE
            WHEN is_legendary = 1 THEN 'yes'
            ELSE 'no'
        END AS is_legendary
    FROM pokemons
)

SELECT 
    generation, 
    name, 
    base_stats_total, 
    is_legendary 
FROM cte 
WHERE rank = 1
ORDER BY generation, rank
"""

duckdb.sql(QUERY).df()

,generation,name,base_stats_total,is_legendary
0,1,Mewtwo,780,yes
1,2,Tyranitar,700,no
2,3,Rayquaza,780,yes
3,4,Arceus,720,yes
4,5,Kyurem,700,yes
5,6,Zygarde,708,yes
6,7,Solgaleo,680,yes
7,7,Lunala,680,yes


If we group Pokemons by type, which is the strongest?


In [27]:
QUERY = """
WITH cte AS (
    SELECT 
        type1 AS primary_type,
        COALESCE(hp, 0) 
            + COALESCE(attack, 0) 
            + COALESCE(defense, 0) 
            + COALESCE(sp_attack, 0) 
            + COALESCE(sp_defense, 0) 
            + COALESCE(speed, 0) 
        AS base_stats_total,
    FROM pokemons
)

SELECT 
    primary_type, 
    ROUND(
        AVG(base_stats_total), 1
    ) AS mean_base_stats_total,
    RANK() OVER (
        ORDER BY mean_base_stats_total DESC
    ) AS rank,
FROM cte
GROUP BY 1
"""

duckdb.sql(QUERY).df()

,primary_type,mean_base_stats_total,rank
0,dragon,522.8,1
1,steel,491.6,2
2,psychic,461.3,3
3,flying,453.3,4
4,fire,450.6,5
5,dark,449.8,6
6,rock,447.3,7
7,electric,436.2,8
8,ghost,434.7,9
9,ice,433.6,10
